<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FEATURE_MONTH = "2026-03"
TARGET_MONTH = "2026-04"
print(f"Feature window: {FEATURE_MONTH}  ->  Target window: {TARGET_MONTH}")
print("Both mid-panel months — never the sealed final month.")

Feature window: 2026-03  ->  Target window: 2026-04
Both mid-panel months — never the sealed final month.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest** (with a Decision Tree as a simpler comparison point).

This follows directly from evidence gathered in earlier weeks: individual signal
correlations with the decline label were all weak (max |0.16| in w02's check),
which suggested the real pattern lives in *combinations* of signals rather than
any single linear relationship — exactly what tree-based ensembles are built to
capture and a logistic regression structurally can't. The starter pipeline
already demonstrated this concretely: random forest reached Precision@50 = 0.740
versus the linear baseline's 0.240 on the starter data.

I'm not using Gradient Boosting here — Random Forest is easier to reason about
and interpret (feature importances, per-tree averaging) for a first real
warehouse model, and the complexity of boosting isn't earning its keep yet at
this stage.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Method: Random Forest (primary), Decision Tree (comparison)")
print("Reason: weak individual correlations (w02) suggest interaction effects, which trees capture and linear models don't.")

Method: Random Forest (primary), Decision Tree (comparison)
Reason: weak individual correlations (w02) suggest interaction effects, which trees capture and linear models don't.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped by client (`client_hash_id`)**, using a GroupShuffleSplit so no
client's pages appear in both train and test.

This matters because pages from the same client likely share patterns (site
structure, content style, existing SEO maturity) that a model could memorize
rather than generalize from. A plain random row split would let the model see
some of a client's pages in training and be tested on other pages from that same
client — an easier, less honest test. Client-holdout matches exactly the
validation design already used in the starter pipeline (`scripts/03_train_model.py`)
and the leakage checklist in the lane guide.

I'm also using a genuine time separation: features come only from March, the
label comes only from April — so no feature was calculated after the decision
point, and the feature window never overlaps the target window.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Split: GroupShuffleSplit grouped by client_hash_id")
print("Time separation: features from", "2026-03", "-> label from", "2026-04")


Split: GroupShuffleSplit grouped by client_hash_id
Time separation: features from 2026-03 -> label from 2026-04


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

build feature frame (prior month):

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features_raw = con.sql(f"""
    SELECT content_hash_id,
           MAX(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position,
           SUM(ga4_sessions) AS sessions,
           SUM(ga4_engaged_sessions) AS engaged_sessions
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={FEATURE_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 50
""").df()
print(f"Feature month rows: {len(features_raw)}")
features_raw.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature month rows: 116114


,content_hash_id,client_hash_id,impressions,clicks,avg_position,sessions,engaged_sessions
0,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.074107,NaN,NaN
1,content_c03ecafd4c999f15,client_62f4a7e64f5e0096,10849.0,22.0,8.240351,NaN,NaN
2,content_e689bc511192751a,client_62f4a7e64f5e0096,61.0,0.0,6.015432,NaN,NaN
3,content_7dbc094b799e05a4,client_62f4a7e64f5e0096,705.0,1.0,5.956862,NaN,NaN
4,content_40b10da45f4c1cb5,client_62f4a7e64f5e0096,50.0,0.0,12.977513,NaN,NaN


build label from the future window:

In [5]:
target_raw = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS future_impressions
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={TARGET_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()
print(f"Target month rows: {len(target_raw)}")

merged = features_raw.merge(target_raw, on="content_hash_id", how="inner")
print(f"Pages present in both months: {len(merged)}")

merged["is_declining"] = (merged["future_impressions"] < 0.8 * merged["impressions"]).astype(int)
merged["ctr"] = merged["clicks"] / merged["impressions"]
print(f"Declining rate: {merged['is_declining'].mean()*100:.1f}%")
merged.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Target month rows: 194760
Pages present in both months: 114993
Declining rate: 51.4%


,content_hash_id,client_hash_id,impressions,clicks,avg_position,sessions,engaged_sessions,future_impressions,is_declining,ctr
0,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.074107,NaN,NaN,27.0,1,0.000000
1,content_c03ecafd4c999f15,client_62f4a7e64f5e0096,10849.0,22.0,8.240351,NaN,NaN,16121.0,0,0.002028
2,content_e689bc511192751a,client_62f4a7e64f5e0096,61.0,0.0,6.015432,NaN,NaN,81.0,0,0.000000
3,content_7dbc094b799e05a4,client_62f4a7e64f5e0096,705.0,1.0,5.956862,NaN,NaN,213.0,1,0.001418
4,content_40b10da45f4c1cb5,client_62f4a7e64f5e0096,50.0,0.0,12.977513,NaN,NaN,23.0,1,0.000000


grouped train/test split:

In [6]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["impressions", "clicks", "avg_position", "sessions", "engaged_sessions", "ctr"]
X = merged[feature_cols].fillna(0)
y = merged["is_declining"]
groups = merged["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
merged_test = merged.iloc[test_idx].reset_index(drop=True)

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Client overlap between train and test: {len(train_clients & test_clients)}")  # should be 0

Train rows: 91850, Test rows: 23143
Client overlap between train and test: 0


train model, build baseline on same test set, compare:

In [9]:
feature_cols = ["impressions", "clicks", "avg_position", "ctr"]
X = merged[feature_cols].fillna(0)
y = merged["is_declining"]
groups = merged["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
merged_test = merged.iloc[test_idx].reset_index(drop=True)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

dt = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)
dt_scores = dt.predict_proba(X_test)[:, 1]

merged_test["position_tier"] = merged_test["avg_position"].apply(tier)
merged_test["tier_expected_ctr"] = merged_test.groupby("position_tier")["ctr"].transform("mean")
merged_test["baseline_score"] = (
    (merged_test["tier_expected_ctr"] - merged_test["ctr"]) * np.log1p(merged_test["impressions"])
)

results = []
for k in (20, 50):
    results.append({"method": "baseline (w04 rule)", "k": k,
                     "precision_at_k": precision_at_k(merged_test["baseline_score"], y_test.values, k)})
    results.append({"method": "decision tree", "k": k,
                     "precision_at_k": precision_at_k(dt_scores, y_test.values, k)})
    results.append({"method": "random forest", "k": k,
                     "precision_at_k": precision_at_k(rf_scores, y_test.values, k)})

results_df = pd.DataFrame(results).pivot(index="method", columns="k", values="precision_at_k")
results_df

k,20,50
method,,
baseline (w04 rule),0.65,0.48
decision tree,0.10,0.20
random forest,0.50,0.40


comparison table interpretation:

**After removing the mostly-NaN GA4 features** (`sessions`, `engaged_sessions`),
using only GSC-based features (impressions, clicks, avg_position, ctr):

| method | Precision@20 | Precision@50 |
|---|---|---|
| baseline (w04 rule) | 0.65 | 0.48 |
| decision tree | 0.10 | 0.20 |
| random forest | 0.50 | 0.40 |

Removing the noisy GA4 columns helped random forest (Precision@20 rose from 0.35
to 0.50), but hurt the decision tree sharply (0.55 → 0.10 at k=20) — a single
tree with only 4 features and max_depth=4 apparently relied heavily on splits
that are no longer available, and a smaller feature set makes a shallow tree
more brittle, not more robust.

**Bottom line, reported honestly: the baseline still wins at both k=20 and
k=50.** This is the opposite of the Week 1-2 starter-pipeline result, and it's a
real, useful finding rather than a failure to paper over. A few honest
candidate explanations:

1. **The label itself may be noisy.** A 20% future-impression drop, month over
   month, may partly reflect normal volatility rather than a genuine,
   persistent decline — the lane guide's Section 7 warns explicitly about
   consolidation, seasonality, and noise being mistaken for real decline, and
   this label doesn't yet rule any of those out.
2. **The baseline rule is already well-matched to this task by construction** —
   it directly encodes "CTR below what this position tier typically gets,"
   which may be a stronger, more targeted signal for this particular label than
   a general-purpose classifier trained on only 4 raw features.
3. **The feature set may simply be too thin.** Four columns (impressions,
   clicks, avg_position, ctr) is a small feature space; the starter pipeline's
   stronger result used a richer set (word count, freshness tiers, engagement
   rate, and more) that this month-to-month warehouse pull doesn't yet include.

This is decision-support evidence that the current model does not yet beat the
baseline on this label and feature set — not evidence that trees never help
here, or that the baseline is definitively better in general.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

feature importance:

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
importances


Random Forest feature importances:


,0
ctr,0.465659
clicks,0.223199
avg_position,0.158847
impressions,0.152295


look at the model's biggest misses on the test set:

In [11]:
merged_test["rf_score"] = rf_scores
merged_test["actual_label"] = y_test.values

false_negatives = merged_test[(merged_test["actual_label"] == 1)].sort_values("rf_score").head(10)
false_positives = merged_test[(merged_test["actual_label"] == 0)].sort_values("rf_score", ascending=False).head(10)

print("Top false negatives (actually declining, model scored them low):")
print(false_negatives[["content_hash_id", "impressions", "avg_position", "ctr", "rf_score"]])

print("\nTop false positives (model scored high, did not actually decline):")
print(false_positives[["content_hash_id", "impressions", "avg_position", "ctr", "rf_score"]])

Top false negatives (actually declining, model scored them low):
                content_hash_id  impressions  avg_position       ctr  rf_score
16649  content_996c52ba57ba223e       4992.0      3.475831  0.004407  0.125641
10026  content_f337dd284c6e83d2      16095.0      2.877057  0.012613  0.164063
16526  content_5e4f0ccdfc0067f5      18876.0      3.261083  0.009589  0.164827
14840  content_747ff6b150f08f84      31941.0      2.673137  0.009674  0.166938
14859  content_20a362f60e0ebc03      20967.0      3.702206  0.012257  0.167004
14839  content_2b9f40230d75e9d6      15014.0      4.213335  0.011190  0.170201
10614  content_cff8ed502bc37690        990.0      4.449429  0.020202  0.170810
11650  content_5a77dbf5671c5a65      19657.0      4.532382  0.010124  0.171029
12016  content_42e142c5cf67e538        612.0      7.748895  0.022876  0.172537
12635  content_fb0b314f4585bb91       1161.0      3.911772  0.015504  0.172877

Top false positives (model scored high, did not actually decline)

**Feature importance:** `ctr` dominates at 46.6%, followed by `clicks` (22.3%),
`avg_position` (15.9%), and `impressions` (15.2%) — fairly evenly split among
the remaining three. CTR being the strongest single signal makes sense given the
label is based on a future impressions drop, and CTR is the feature most
directly tied to how a page is actually performing relative to its own exposure.

**Where the model is wrong:**

False negatives (actually declining pages the model scored low, ~0.12-0.17)
share a clear pattern: they mostly have **good position (2.7-4.5, one at 7.7)
and reasonably low but non-trivial CTR (0.009-0.023)** — a profile the model
reads as "healthy" (good rank, plausible CTR) even though the page went on to
lose future impressions. The model appears to lean heavily on CTR and position
together as "things look fine," and misses pages whose future decline isn't
visible yet in this month's snapshot — consistent with the idea that a single
prior month may not carry enough signal to anticipate a future drop.

False positives (model scored high, ~0.69-0.71, but didn't actually decline)
share an even sharper pattern: **extremely low or literally zero CTR**
(0.000000-0.001443) at moderate-to-high impressions. Two rows even sit at
avg_position 80+ with near-zero CTR — pages that look like obvious "problem"
candidates by the model's logic, yet held steady rather than declining further.
This suggests the model treats "already very low CTR" as a strong decline
signal, but for pages already at a low baseline, there may be little room left
to fall — the label (a *relative* future drop) doesn't distinguish "already bad,
staying bad" from "was fine, now declining."

**Why the baseline may still be winning:** the false-positive pattern is
revealing — the model's top feature (CTR) is also central to the w04 baseline
rule, but the baseline compares CTR *within position tier*, while the random
forest's CTR importance isn't tier-adjusted. That may explain why the baseline
avoids some of these false positives: it wouldn't flag an already-low-CTR page
at position 80+ as urgent, since low CTR is *expected* at that tier — while the
random forest, without that normalization built in, appears to treat low CTR as
alarming regardless of position.

**Careful words:** this is an observed, directional pattern from one train/test
split on one month-pair (March→April), not proof that the model structurally
can't work — it points at a specific, fixable gap (position-normalized CTR as a
feature, or a less volatile label) rather than a dead end.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.